In [10]:
%use ktor-client, serialization, dataframe, datetime, coroutines

In [11]:
import io.ktor.client.*
import io.ktor.client.engine.cio.*
import io.ktor.client.plugins.*
import io.ktor.client.plugins.contentnegotiation.*
import io.ktor.client.request.*
import io.ktor.client.request.forms.*
import io.ktor.client.statement.*
import io.ktor.http.*
import io.ktor.serialization.kotlinx.json.*

import kotlinx.coroutines.*
import kotlinx.serialization.*
import kotlinx.serialization.json.*

import org.jetbrains.kotlinx.dataframe.api.*
import java.util.Base64

### Config

In [12]:
val json = Json {
    ignoreUnknownKeys = true
    isLenient = true
    prettyPrint = true
}

val http = HttpClient(CIO) {
    install(ContentNegotiation) { json(json) }
    install(HttpTimeout) {
        requestTimeoutMillis = 15_000
        connectTimeoutMillis = 10_000
        socketTimeoutMillis = 15_000
    }
    expectSuccess = false
}

data class Env(
    val keycloakBase: String = "http://localhost:7766",
    val realm: String = "flagforge",
    val clientId: String = "flagforge-frontend",
    val gatewayBase: String = "http://localhost:8081",
)

val env = Env()

### utils & dtos

In [13]:
@Serializable
data class TokenResponse(
    @SerialName("access_token") val accessToken: String,
    @SerialName("expires_in") val expiresIn: Long? = null,
    @SerialName("token_type") val tokenType: String? = null,
)

data class HttpResult(
    val step: String,
    val method: String,
    val url: String,
    val status: Int,
    val body: String
)

suspend fun requestRaw(
    step: String,
    method: HttpMethod,
    url: String,
    headers: Map<String, String> = emptyMap(),
    contentType: ContentType? = null,
    bodyText: String? = null,
): HttpResult {
    val resp = http.request(url) {
        this.method = method
        headers.forEach { (k, v) -> header(k, v) }
        if (contentType != null) contentType(contentType)
        if (bodyText != null) setBody(bodyText)
    }
    return HttpResult(step, method.value, url, resp.status.value, resp.bodyAsText())
}

fun HttpResult.row(): Map<String, Any?> = mapOf(
    "step" to step,
    "method" to method,
    "status" to status,
    "url" to url,
    "body" to body.take(500)
)

fun assertStatus(res: HttpResult, expected: Int): HttpResult {
    require(res.status == expected) {
        "Expected HTTP $expected but got ${res.status} for ${res.url}\n${res.body}"
    }
    return res
}

fun assertStatusIn(res: HttpResult, expected: Set<Int>): HttpResult {
    require(res.status in expected) {
        "Expected HTTP in $expected but got ${res.status} for ${res.url}\n${res.body}"
    }
    return res
}

In [14]:
suspend fun gwCall(
    step: String,
    method: HttpMethod,
    path: String,
    token: String? = null,
    actorId: String? = null,
    jsonBody: String? = null
): HttpResult {
    val headers = buildMap {
        if (token != null) put("Authorization", "Bearer $token")
        if (actorId != null) put("X-Actor-Id", actorId)
    }
    return requestRaw(
        step = step,
        method = method,
        url = "${env.gatewayBase}$path",
        headers = headers,
        contentType = if (jsonBody != null) ContentType.Application.Json else null,
        bodyText = jsonBody
    )
}


# Keycloak Admin

In [15]:
fun decodeJwtPayload(token: String): JsonObject {
    val parts = token.split(".")
    require(parts.size >= 2) { "Not a JWT" }
    val payloadB64 = parts[1]
    val decoded = String(Base64.getUrlDecoder().decode(payloadB64))
    return json.parseToJsonElement(decoded).jsonObject
}

fun extractRoles(payload: JsonObject): List<String> {
    val realmRoles = payload["realm_access"]?.jsonObject
        ?.get("roles")?.jsonArray?.map { it.jsonPrimitive.content }.orEmpty()

    val clientRoles = payload["resource_access"]?.jsonObject?.entries
        ?.flatMap { (client, v) ->
            val roles = v.jsonObject["roles"]?.jsonArray?.map { it.jsonPrimitive.content }.orEmpty()
            roles.map { r -> "$client:$r" }
        }.orEmpty()

    return (realmRoles + clientRoles).distinct()
}

suspend fun token(username: String, password: String): String {
    val url = "${env.keycloakBase}/realms/${env.realm}/protocol/openid-connect/token"
    val form = Parameters.build {
        append("grant_type", "password")
        append("client_id", env.clientId)
        append("username", username)
        append("password", password)
    }.formUrlEncode()

    val res = requestRaw(
        step = "Keycloak token for $username",
        method = HttpMethod.Post,
        url = url,
        contentType = ContentType.Application.FormUrlEncoded,
        bodyText = form
    )
    assertStatus(res, 200)
    return json.decodeFromString<TokenResponse>(res.body).accessToken
}

val adminToken = runBlocking { token("ff-admin", "ff") }
val userToken  = runBlocking { token("ff-user", "ff") }

val tokensDf = listOf(
    mapOf(
        "who" to "admin",
        "preferred_username" to decodeJwtPayload(adminToken)["preferred_username"]?.jsonPrimitive?.contentOrNull,
        "roles" to extractRoles(decodeJwtPayload(adminToken)).joinToString(", ")
    ),
    mapOf(
        "who" to "user",
        "preferred_username" to decodeJwtPayload(userToken)["preferred_username"]?.jsonPrimitive?.contentOrNull,
        "roles" to extractRoles(decodeJwtPayload(userToken)).joinToString(", ")
    )
).toDataFrame()

tokensDf

value
"{who=admin, preferred_username=ff-adm..."
"{who=user, preferred_username=ff-user..."


# 401 Создание проекта без токена

In [16]:
val createProjectJson = buildJsonObject {
    put("key", "sec-demo")
    put("name", "Security Demo")
}.toString()

val r1 = runBlocking {
    gwCall(
        step = "NO TOKEN -> control create project",
        method = HttpMethod.Post,
        path = "/api/v1/control/projects",
        token = null,
        actorId = "anonymous",
        jsonBody = createProjectJson
    )
}

listOf(r1).toDataFrame()

step,method,url,status,body
NO TOKEN -> control create project,POST,http://localhost:8081/api/v1/control/...,401,


### 403 Создание проекта с валидным токеном юзера

In [17]:
val r2 = runBlocking {
    gwCall(
        step = "USER TOKEN -> control create project",
        method = HttpMethod.Post,
        path = "/api/v1/control/projects",
        token = userToken,
        actorId = "ff-user",
        jsonBody = createProjectJson
    )
}

listOf(r2).toDataFrame()

step,method,url,status,body
USER TOKEN -> control create project,POST,http://localhost:8081/api/v1/control/...,403,


# 200 Создание проекта с токеном админа

In [18]:
val r3 = runBlocking {
    gwCall(
        step = "ADMIN TOKEN -> control create project",
        method = HttpMethod.Post,
        path = "/api/v1/control/projects",
        token = adminToken,
        actorId = "ff-admin",
        jsonBody = createProjectJson
    )
}

listOf(r3).toDataFrame()

step,method,url,status,body
ADMIN TOKEN -> control create project,POST,http://localhost:8081/api/v1/control/...,202,"{""apiVersion"":""v1"",""status"":""ACCEPTED..."
